In [2]:
import os
import sys
import json
import urllib.request
import subprocess
from pathlib import Path

ARTICLE_ID = "28804448"
FILE_ID = 53696243
DOWNLOAD_DEST = "MmodalFire_dataset.rar"

def download_file(url, dest, description="File"):
    print(f"\n🚀 Downloading {description}...")
    try:
        req = urllib.request.Request(
            url,
            headers={'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}
        )
        
        with urllib.request.urlopen(req) as response:
            file_size_bytes = int(response.headers.get('Content-Length', 0))
            if file_size_bytes > 0:
                print(f"📦 Expected File Size: {file_size_bytes / (1024**2):.2f} MB")
            
            with open(dest, 'wb') as out_file:
                chunk_size = 1024 * 1024 * 4  # 4 MB chunks
                downloaded = 0
                last_percent = -1
                
                while True:
                    chunk = response.read(chunk_size)
                    if not chunk:
                        break
                    out_file.write(chunk)
                    downloaded += len(chunk)
                    
                    if file_size_bytes > 0:
                        percent = int((downloaded / file_size_bytes) * 100)
                        if percent % 10 == 0 and percent != last_percent:
                            print(f"⬇️ Progress: {percent}% (~{downloaded / (1024**2):.2f} MB)")
                            last_percent = percent
                    else:
                        print(".", end="", flush=True)
                        
            print(f"\n✅ Download Complete: {dest}")
            return True
    except Exception as e:
        print(f"❌ Download failed: {e}")
        return False

def compile_official_unrar():
    print("\n🛠️ Building RAR5 compatibility engine from source...")
    unrar_src_url = "https://www.rarlab.com/rar/unrarsrc-7.2.4.tar.gz"
    unrar_tar = "unrarsrc.tar.gz"
    
    if download_file(unrar_src_url, unrar_tar, "official unrar source"):
        try:
            print("📦 Unpacking compiler files...")
            os.system(f"tar -xzf {unrar_tar}")
            
            print("⚡ Compiling unrar binary (this takes under 10 seconds)...")
            # Compile with multiple cores for speed
            result = subprocess.run(
                ["make", "-j", "4", "-f", "makefile"], 
                cwd="./unrar", 
                stdout=subprocess.DEVNULL, 
                stderr=subprocess.PIPE
            )
            
            if result.returncode == 0 and os.path.exists("./unrar/unrar"):
                print("🎉 Compilation successful! Native RAR5 codec loaded.")
                return os.path.abspath("./unrar/unrar")
            else:
                print(f"⚠️ Compiler exited with code {result.returncode}. Error: {result.stderr.decode()}")
        except Exception as e:
            print(f"❌ Failed to build extractor: {e}")
            
    print("⚠️ Falling back to system standard tools...")
    return "unrar"

print("🔍 Querying Figshare API for direct download link...")
try:
    api_url = f"https://api.figshare.com/v2/articles/{ARTICLE_ID}"
    req = urllib.request.Request(
        api_url, 
        headers={'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}
    )
    with urllib.request.urlopen(req) as response:
        data = json.loads(response.read().decode())
    
    download_url = None
    for f in data.get('files', []):
        if f.get('id') == FILE_ID:
            download_url = f.get('download_url')
            break

    if not download_url:
        download_url = f"https://api.figshare.com/v2/file/download/{FILE_ID}"

except Exception as e:
    print(f"⚠️ Figshare API query warning: {e}. Using fallback route...")
    download_url = f"https://api.figshare.com/v2/file/download/{FILE_ID}"

if not os.path.exists(DOWNLOAD_DEST) or os.path.getsize(DOWNLOAD_DEST) < 4000000000:
    download_file(download_url, DOWNLOAD_DEST, "multimodal dataset (RAR)")
else:
    print(f"✨ Found existing {DOWNLOAD_DEST} of healthy size. Skipping download.")

unrar_bin = compile_official_unrar()

print("\n📂 Setting up extraction folders...")
extraction_dir = Path("./mmodal_fire_extracted")
extraction_dir.mkdir(parents=True, exist_ok=True)

print("📦 Extracting archive using newly compiled RAR5-compatible unrar...")
try:
    # Use -y to auto-overwrite/accept prompts and run in extract-with-paths mode (x)
    exit_code = os.system(f'"{unrar_bin}" x -y {DOWNLOAD_DEST} {extraction_dir} > /dev/null')
    
    if exit_code == 0:
        print("\n🎉 Extraction completely successful without errors!")
        print("\n🔍 Top-level directories extracted:")
        print(os.listdir(extraction_dir))
        
        # Stream subdirectories to explore structure
        raw_data_dir = extraction_dir / "原始数据"
        if raw_data_dir.exists():
            print(f"📁 Inside '{raw_data_dir.name}': {os.listdir(raw_data_dir)}")
    else:
        print(f"❌ Extraction process returned failure status code: {exit_code}")
except Exception as e:
    print(f"❌ Extraction process completely failed: {e}")

🔍 Querying Figshare API for direct download link...

🚀 Downloading multimodal dataset (RAR)...
📦 Expected File Size: 4700.86 MB
⬇️ Progress: 0% (~4.00 MB)
⬇️ Progress: 10% (~472.00 MB)
⬇️ Progress: 20% (~944.00 MB)
⬇️ Progress: 30% (~1412.00 MB)
⬇️ Progress: 40% (~1884.00 MB)
⬇️ Progress: 50% (~2352.00 MB)
⬇️ Progress: 60% (~2824.00 MB)
⬇️ Progress: 70% (~3292.00 MB)
⬇️ Progress: 80% (~3764.00 MB)
⬇️ Progress: 90% (~4232.00 MB)
⬇️ Progress: 100% (~4700.86 MB)

✅ Download Complete: MmodalFire_dataset.rar

🛠️ Building RAR5 compatibility engine from source...

🚀 Downloading official unrar source...
📦 Expected File Size: 0.26 MB
⬇️ Progress: 100% (~0.26 MB)

✅ Download Complete: unrarsrc.tar.gz
📦 Unpacking compiler files...
⚡ Compiling unrar binary (this takes under 10 seconds)...
🎉 Compilation successful! Native RAR5 codec loaded.

📂 Setting up extraction folders...
📦 Extracting archive using newly compiled RAR5-compatible unrar...

🎉 Extraction completely successful without errors!

🔍 To

In [4]:
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 59.0 MB/s eta 0:00:00


In [13]:
import os
import sys
import json
import urllib.request
import subprocess
from pathlib import Path

ARTICLE_ID = "28804448"
FILE_ID = 53696243
DOWNLOAD_DEST = "MmodalFire_dataset.rar"

def download_file(url, dest, description="File"):
    print(f"\n🚀 Downloading {description}...")
    try:
        req = urllib.request.Request(
            url,
            headers={'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}
        )
        
        with urllib.request.urlopen(req) as response:
            file_size_bytes = int(response.headers.get('Content-Length', 0))
            if file_size_bytes > 0:
                print(f"📦 Expected File Size: {file_size_bytes / (1024**2):.2f} MB")
            
            with open(dest, 'wb') as out_file:
                chunk_size = 1024 * 1024 * 4  # 4 MB chunks
                downloaded = 0
                last_percent = -1
                
                while True:
                    chunk = response.read(chunk_size)
                    if not chunk:
                        break
                    out_file.write(chunk)
                    downloaded += len(chunk)
                    
                    if file_size_bytes > 0:
                        percent = int((downloaded / file_size_bytes) * 100)
                        if percent % 10 == 0 and percent != last_percent:
                            print(f"⬇️ Progress: {percent}% (~{downloaded / (1024**2):.2f} MB)")
                            last_percent = percent
                    else:
                        print(".", end="", flush=True)
                        
            print(f"\n✅ Download Complete: {dest}")
            return True
    except Exception as e:
        print(f"❌ Download failed: {e}")
        return False

def compile_official_unrar():
    print("\n🛠️ Building RAR5 compatibility engine from source...")
    unrar_src_url = "https://www.rarlab.com/rar/unrarsrc-7.2.4.tar.gz"
    unrar_tar = "unrarsrc.tar.gz"
    
    if download_file(unrar_src_url, unrar_tar, "official unrar source"):
        try:
            print("📦 Unpacking compiler files...")
            os.system(f"tar -xzf {unrar_tar}")
            
            print("⚡ Compiling unrar binary (this takes under 10 seconds)...")
            # Compile with multiple cores for speed
            result = subprocess.run(
                ["make", "-j", "4", "-f", "makefile"], 
                cwd="./unrar", 
                stdout=subprocess.DEVNULL, 
                stderr=subprocess.PIPE
            )
            
            if result.returncode == 0 and os.path.exists("./unrar/unrar"):
                print("🎉 Compilation successful! Native RAR5 codec loaded.")
                return os.path.abspath("./unrar/unrar")
            else:
                print(f"⚠️ Compiler exited with code {result.returncode}. Error: {result.stderr.decode()}")
        except Exception as e:
            print(f"❌ Failed to build extractor: {e}")
            
    print("⚠️ Falling back to system standard tools...")
    return "unrar"

print("🔍 Querying Figshare API for direct download link...")
try:
    api_url = f"https://api.figshare.com/v2/articles/{ARTICLE_ID}"
    req = urllib.request.Request(
        api_url, 
        headers={'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}
    )
    with urllib.request.urlopen(req) as response:
        data = json.loads(response.read().decode())
    
    download_url = None
    for f in data.get('files', []):
        if f.get('id') == FILE_ID:
            download_url = f.get('download_url')
            break

    if not download_url:
        download_url = f"https://api.figshare.com/v2/file/download/{FILE_ID}"

except Exception as e:
    print(f"⚠️ Figshare API query warning: {e}. Using fallback route...")
    download_url = f"https://api.figshare.com/v2/file/download/{FILE_ID}"

if not os.path.exists(DOWNLOAD_DEST) or os.path.getsize(DOWNLOAD_DEST) < 4000000000:
    download_file(download_url, DOWNLOAD_DEST, "multimodal dataset (RAR)")
else:
    print(f"✨ Found existing {DOWNLOAD_DEST} of healthy size. Skipping download.")

unrar_bin = compile_official_unrar()

print("\n📂 Setting up extraction folders...")
extraction_dir = Path("./mmodal_fire_extracted")
extraction_dir.mkdir(parents=True, exist_ok=True)

print("📦 Extracting archive using newly compiled RAR5-compatible unrar...")
try:
    # Use -y to auto-overwrite/accept prompts and run in extract-with-paths mode (x)
    exit_code = os.system(f'"{unrar_bin}" x -y {DOWNLOAD_DEST} {extraction_dir} > /dev/null')
    
    if exit_code == 0:
        print("\n🎉 Extraction completely successful without errors!")
        print("\n🔍 Top-level directories extracted:")
        print(os.listdir(extraction_dir))
        
        # Stream subdirectories to explore structure
        raw_data_dir = extraction_dir / "原始数据"
        if raw_data_dir.exists():
            print(f"📁 Inside '{raw_data_dir.name}': {os.listdir(raw_data_dir)}")
    else:
        print(f"❌ Extraction process returned failure status code: {exit_code}")
except Exception as e:
    print(f"❌ Extraction process completely failed: {e}")

🔍 Querying Figshare API for direct download link...

🚀 Downloading multimodal dataset (RAR)...
📦 Expected File Size: 4700.86 MB
⬇️ Progress: 0% (~4.00 MB)
⬇️ Progress: 10% (~472.00 MB)
⬇️ Progress: 20% (~944.00 MB)
⬇️ Progress: 30% (~1412.00 MB)
⬇️ Progress: 40% (~1884.00 MB)
⬇️ Progress: 50% (~2352.00 MB)
⬇️ Progress: 60% (~2824.00 MB)
⬇️ Progress: 70% (~3292.00 MB)
⬇️ Progress: 80% (~3764.00 MB)
⬇️ Progress: 90% (~4232.00 MB)
⬇️ Progress: 100% (~4700.86 MB)

✅ Download Complete: MmodalFire_dataset.rar

🛠️ Building RAR5 compatibility engine from source...

🚀 Downloading official unrar source...
📦 Expected File Size: 0.26 MB
⬇️ Progress: 100% (~0.26 MB)

✅ Download Complete: unrarsrc.tar.gz
📦 Unpacking compiler files...
⚡ Compiling unrar binary (this takes under 10 seconds)...
🎉 Compilation successful! Native RAR5 codec loaded.

📂 Setting up extraction folders...
📦 Extracting archive using newly compiled RAR5-compatible unrar...

🎉 Extraction completely successful without errors!

🔍 To

In [ ]:
import os
import cv2
import random
import pandas as pd
from pathlib import Path

source_video_dir = Path("./mmodal_fire_extracted/原始数据/Video")
dataset_base_dir = Path("./fire_classification_dataset")
excel_mapping_path = Path("./mmodal_fire_extracted/原始数据/SETTINGS OF THE COMBUSTION EXPERIMENT FOR DATA COLLECTION.xlsx")

classes = ["fire", "smoke", "steam", "normal"]
for split in ["train", "val"]:
    for cls in classes:
        (dataset_base_dir / split / cls).mkdir(parents=True, exist_ok=True)

# 1. READ EXCEL
df = pd.read_excel(excel_mapping_path, sheet_name=0)

# 2. DISCOVER FILES
video_files = []
for root, dirs, files in os.walk(source_video_dir):
    for f in files:
        if f.lower().endswith(('.mp4', '.avi', '.mov')):
            video_files.append((Path(root) / f, f))

# 3. MAPPING LOGIC (Index Fallback with Verbose Print)
category_map = {cls: [] for cls in classes}
sorted_videos = sorted(video_files, key=lambda x: x[1])

print(f"🧬 Mapping {len(sorted_videos)} videos using Index Fallback...")
for idx, (path, filename) in enumerate(sorted_videos):
    # Determine class based on row index from Excel
    row = df.iloc[idx % len(df)]
    mat = str(row.get("Material", "")).lower()
    
    if any(k in mat for k in ["heptane", "polyurethane", "fire"]): cls = "fire"
    elif any(k in mat for k in ["cotton", "smoke", "smoulder"]): cls = "smoke"
    elif any(k in mat for k in ["humidifier", "dry ice", "steam"]): cls = "steam"
    else: cls = "normal"
    
    category_map[cls].append((path, filename))
    print(f"   Mapped {filename} -> {cls}")

# 4. EXTRACTION (Verbose Print)
def extract_frames(video_list, split_name):
    print(f"\n📸 Extracting frames for '{split_name}'...")
    for path, name, cls in video_list:
        print(f"   -> Processing: {name} ({cls})") # This print is the "heartbeat"
        cap = cv2.VideoCapture(str(path))
        fps = cap.get(cv2.CAP_PROP_FPS) or 25
        frame_count = 0
        while True:
            ret, frame = cap.read()
            if not ret: break
            if frame_count % int(fps * 2) == 0: # 1 frame every 2s
                cv2.imwrite(str(dataset_base_dir / split_name / cls / f"{Path(name).stem}_{frame_count}.jpg"), cv2.resize(frame, (224, 224)))
            frame_count += 1
        cap.release()

# SPLIT
train_vids, val_vids = [], []
for cls, files in category_map.items():
    split = int(len(files) * 0.8)
    for f in files[:split]: train_vids.append((f[0], f[1], cls))
    for f in files[split:]: val_vids.append((f[0], f[1], cls))

extract_frames(train_vids, "train")
extract_frames(val_vids, "val")
print("\n✅ Extraction complete. Ready for Training.")

🧬 Mapping 28 videos using Index Fallback...
   Mapped Downtest_ch01_00000000001000100.mp4 -> smoke
   Mapped Downtest_ch01_00000000002000000.mp4 -> smoke
   Mapped Downtest_ch01_00000000002000100.mp4 -> smoke
   Mapped Downtest_ch01_00000000002000300.mp4 -> smoke
   Mapped Downtest_ch01_00000000003000000.mp4 -> smoke
   Mapped Downtest_ch01_00000000004000200.mp4 -> smoke
   Mapped Downtest_ch01_00000000005000000.mp4 -> smoke
   Mapped Downtest_ch01_00000000005000100.mp4 -> smoke
   Mapped Downtest_ch01_00000000006000000.mp4 -> smoke
   Mapped Downtest_ch01_00000000007000100.mp4 -> fire
   Mapped Downtest_ch01_00000000008000000.mp4 -> fire
   Mapped Downtest_ch01_00000000009000000.mp4 -> fire
   Mapped Downtest_ch01_00000000010000000.mp4 -> fire
   Mapped Downtest_ch01_00000000011000000.mp4 -> fire
   Mapped Downtest_ch01_00000000011000100.mp4 -> fire
   Mapped Downtest_ch01_00000000012000000.mp4 -> fire
   Mapped Downtest_ch01_00000000013000000.mp4 -> fire
   Mapped Downtest_ch01_00000